# Generación Musical y Audio Creativo con IA

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/voz-audio/04-generacion-audio-creativo.ipynb)

Generación de música con MusicGen (Meta, open-source), efectos de sonido con ElevenLabs SFX y podcast automatizado con Claude + TTS.

In [ ]:
!pip install anthropic openai -q
# Para MusicGen local (requiere ~2GB descarga):
# pip install transformers torch scipy

In [ ]:
import anthropic
import openai

anthropic_client = anthropic.Anthropic()
openai_client = openai.OpenAI()

## 1. Claude como generador de prompts musicales

In [ ]:
def generar_prompt_musical(descripcion: str, estilo: str = 'background') -> str:
    """
    Usa Claude para generar un prompt optimizado para herramientas de música IA.
    estilo: 'background', 'podcast', 'energetic', 'emotional', 'corporate'
    """
    estilos_contexto = {
        'background': 'música de fondo sutil que no distraiga',
        'podcast': 'intro/outro de podcast, profesional y moderno',
        'energetic': 'música enérgica y motivacional',
        'emotional': 'música emocional para storytelling',
        'corporate': 'música corporativa para presentaciones',
    }
    contexto_estilo = estilos_contexto.get(estilo, estilo)
    
    response = anthropic_client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=100,
        messages=[{
            'role': 'user',
            'content': f"""Genera un prompt de 15-20 palabras en inglés para MusicGen.
Contexto: {descripcion}
Estilo buscado: {contexto_estilo}
Incluye: género, instrumentos, tempo/energía.
Solo el prompt, sin explicación."""
        }],
    )
    return response.content[0].text.strip().strip('"')

# Ejemplos de distintos casos de uso
casos = [
    ('Tutorial de Python para principiantes', 'background'),
    ('Podcast de noticias tecnológicas', 'podcast'),
    ('Presentación de resultados trimestrales', 'corporate'),
    ('Documental sobre naturaleza', 'emotional'),
]

print('Prompts musicales generados por Claude:')
for descripcion, estilo in casos:
    prompt = generar_prompt_musical(descripcion, estilo)
    print(f'\n📌 {descripcion} [{estilo}]')
    print(f'   → {prompt}')

## 2. MusicGen local (requiere instalación adicional)

In [ ]:
# Clase MusicGen — descomentar y ejecutar si tienes transformers instalado

MUSICGEN_DISPONIBLE = False
try:
    from transformers import MusicgenForConditionalGeneration, AutoProcessor
    MUSICGEN_DISPONIBLE = True
except ImportError:
    pass

if MUSICGEN_DISPONIBLE:
    import torch
    import scipy.io.wavfile
    import numpy as np
    
    class GeneradorMusica:
        def __init__(self, modelo='facebook/musicgen-small'):
            print(f'Cargando {modelo}...')
            self.processor = AutoProcessor.from_pretrained(modelo)
            self.model = MusicgenForConditionalGeneration.from_pretrained(modelo)
            self.model.eval()
            self.sample_rate = self.model.config.audio_encoder.sampling_rate
        
        def generar(self, prompt: str, segundos: int = 10, ruta: str = '/tmp/musica.wav') -> str:
            inputs = self.processor(text=[prompt], padding=True, return_tensors='pt')
            with torch.no_grad():
                audio = self.model.generate(**inputs, max_new_tokens=segundos * 50)
            audio_np = audio[0, 0].numpy()
            scipy.io.wavfile.write(ruta, self.sample_rate, (audio_np * 32767).astype(np.int16))
            print(f'✅ {segundos}s de música generados → {ruta}')
            return ruta
    
    gen = GeneradorMusica()
    prompt = generar_prompt_musical('Tutorial de IA para empresas', 'background')
    gen.generar(prompt, segundos=10)
else:
    print('MusicGen no instalado. Ejecuta: pip install transformers torch scipy')
    print('\nPrompt que se usaría con MusicGen:')
    print(' ', generar_prompt_musical('Tutorial de IA para empresas', 'background'))

## 3. Pipeline completo: guión de podcast + narración

In [ ]:
def generar_podcast_minimo(tema: str, duracion_min: int = 2) -> dict:
    """
    Pipeline: Claude genera guión → OpenAI TTS narra → guarda audio.
    Versión compacta para demo (guión corto para minimizar costes TTS).
    """
    # 1. Guión con Claude
    response = anthropic_client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=600,
        messages=[{
            'role': 'user',
            'content': f"""Escribe un guión de podcast de {duracion_min} minuto(s) sobre: {tema}
Estructura: intro (20s) → 2 puntos principales (25s cada uno) → conclusión (10s)
Tono conversacional, sin markdown, sin mencionar que es un guión."""
        }],
    )
    guion = response.content[0].text
    
    # 2. TTS
    audio = openai_client.audio.speech.create(
        model='tts-1-hd',
        voice='onyx',
        input=guion,
    )
    ruta_audio = '/tmp/podcast_demo.mp3'
    audio.stream_to_file(ruta_audio)
    
    # 3. Prompt musical (para cuando añadas la música)
    prompt_musical = generar_prompt_musical(tema, 'podcast')
    
    return {
        'tema': tema,
        'guion': guion,
        'audio': ruta_audio,
        'prompt_musical': prompt_musical,
        'chars': len(guion),
        'coste_estimado_usd': round(len(guion) / 1_000_000 * 15, 4),
    }

resultado = generar_podcast_minimo('Cómo los LLMs están cambiando el desarrollo de software', duracion_min=1)

print(f'Tema: {resultado["tema"]}')
print(f'Caracteres del guión: {resultado["chars"]}')
print(f'Coste estimado TTS: ${resultado["coste_estimado_usd"]}')
print(f'Audio guardado: {resultado["audio"]}')
print(f'Prompt musical sugerido: {resultado["prompt_musical"]}')
print(f'\nExtracto del guión:')
print(resultado['guion'][:300] + '...')

## 4. Resumen: herramientas de audio IA

In [ ]:
import pandas as pd

herramientas = pd.DataFrame({
    'Herramienta': ['MusicGen (Meta)', 'Suno', 'ElevenLabs SFX', 'OpenAI TTS', 'Whisper'],
    'Tipo': ['Música', 'Música + letra', 'Efectos sonido', 'Síntesis voz', 'Transcripción'],
    'Open Source': ['✅', '❌', '❌', '❌', '✅'],
    'API': ['❌ (local)', '✅', '✅', '✅', '✅'],
    'Uso comercial': ['CC-BY-NC', 'Ver licencia', '✅', '✅', '✅'],
    'Mejor para': [
        'Música fondo local',
        'Canciones completas',
        'Sonidos UI/UX',
        'Narración y asistentes',
        'Transcripción vídeos',
    ],
})

print(herramientas.to_string(index=False))